# Orbit Wars — V2 OrbitNet Colab Training

GPU-accelerated training of the V2 OrbitNet agent on Google Colab with Google Drive persistence.

**V2 Architecture:** Single forward pass per game step. Self-attention over all 40 planets with pairwise output head producing a 40×41 allocation matrix. Softmax probability = ship fraction.

**Features tested in this notebook:**
- **Behavioral Cloning (BC)** — pretrain on apex demonstrations before PPO
- **Parallel rollout collection** — subprocess workers for faster training
- **Viability masking** — constrain actions to viable fleet sends only
- **Fleet size enforcement** — prevent undersized fleets
- **Offline data collection** — save demo buffer + rollout data for future training

**Pipeline:**
1. Setup (install deps, mount Drive, clone repo)
2. GPU verification
3. Configure experiment (BC + PPO settings)
4. Train (BC pretrain → PPO with decaying imitation loss)
5. Save offline data for future training
6. Generate submission
7. Evaluate against baselines
8. TensorBoard monitoring

In [ ]:
#@title 1. Setup: Mount Drive, Clone Repo, Install Dependencies

import os, sys

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Persistent output directory on Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/orbit_wars_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/submissions', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/offline_data', exist_ok=True)
print(f'Drive output dir: {DRIVE_OUTPUT}')

# ── Clone or update repo ────────────────────────────────────────────────────
REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT THIS
REPO_DIR = '/content/orbit_wars'

if os.path.exists(REPO_DIR):
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')

# ── Install dependencies ────────────────────────────────────────────────────
os.system('pip install -q --upgrade "kaggle-environments>=1.28.0"')
os.system('pip install -q pyyaml tensorboard')

print('\nSetup complete.')

In [ ]:
#@title 2. GPU Verification

import torch

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU detected! Training will be slow.')
    print('Go to Runtime > Change runtime type > GPU')
    DEVICE = 'cpu'

print(f'\nUsing device: {DEVICE}')

## Configuration

Edit the cell below to configure your experiment. Two modes available:

**BC + PPO (recommended):** Behavioral cloning pretrain from apex demonstrations, then PPO fine-tuning with decaying imitation loss. Gives the model a strong starting point.

**PPO only:** Train from random weights (slower learning, set `ENABLE_BC = False`).

GPU overrides are applied for Colab's hardware (more envs, longer rollouts, parallel workers).

In [ ]:
#@title 3. Experiment Config

from v2.config import load_v2_config

# ── Choose mode ──────────────────────────────────────────────────────────
ENABLE_BC = True   # Set False to skip BC pretrain and run pure PPO

base_config = 'configs/v2_bc.yaml' if ENABLE_BC else 'configs/v2_default.yaml'
cfg = load_v2_config(base_config)

# ── GPU Colab overrides ───────────────────────────────────────────────────
cfg.device = DEVICE
cfg.ppo.num_envs = 4
cfg.ppo.rollout_steps = 128
cfg.ppo.total_updates = 5000
cfg.ppo.num_workers = 4           # parallel rollout collection (4 subprocess workers)
cfg.ppo.ent_coef = 0.01           # lower entropy for faster policy specialization

cfg.eval.eval_every = 250
cfg.eval.eval_games = 10

# Point outputs to Google Drive for persistence
cfg.save_dir = f'{DRIVE_OUTPUT}/checkpoints'
cfg.log_dir = f'{DRIVE_OUTPUT}/logs'

# ── BC settings (only matter if ENABLE_BC=True) ──────────────────────────
if ENABLE_BC:
    cfg.imitation.enabled = True
    cfg.imitation.bc_expert = 'apex'        # expert agent to clone
    cfg.imitation.bc_games = 200            # demo games to collect
    cfg.imitation.bc_demo_opponent = 'random'  # expert plays against this
    cfg.imitation.bc_epochs = 50            # supervised pretraining epochs
    cfg.imitation.bc_lr = 0.001
    cfg.imitation.bc_batch_size = 256
    cfg.imitation.coef_start = 0.5          # initial BC loss weight during PPO
    cfg.imitation.coef_decay_updates = 1000 # linear decay to 0 over this many updates
    cfg.imitation.distilled_opponent = True  # use BC model as training opponent
    cfg.ppo.lr = 0.0001                     # lower LR since starting from BC weights

# ── Print config summary ─────────────────────────────────────────────────
from v2.model import OrbitNet
n_params = sum(p.numel() for p in OrbitNet(cfg.model).parameters())

print(f'Run: {cfg.run_name}')
print(f'Device: {cfg.device}')
print(f'OrbitNet params: {n_params:,}')
print(f'Updates: {cfg.ppo.total_updates}')
print(f'Envs: {cfg.ppo.num_envs}, rollout_steps: {cfg.ppo.rollout_steps}, '
      f'workers: {cfg.ppo.num_workers}')
print(f'LR: {cfg.ppo.lr}, entropy_coef: {cfg.ppo.ent_coef}')
print(f'Opponent: {cfg.opponent}')
print(f'Reward: mode={cfg.reward.reward_mode}, ship_coef={cfg.reward.dense_ship_coef}')
print(f'Early prod bonus: {1 + cfg.reward.early_prod_bonus}x at step 0, '
      f'decays to 1x at step {cfg.reward.early_prod_bonus_steps}')
print(f'4-player prob: {cfg.four_player_prob}')
print(f'Rule-based decay: {cfg.rule_based_prob_start} -> {cfg.rule_based_prob_end} '
      f'over {cfg.rule_based_decay_updates} updates')
print(f'Eval: every {cfg.eval.eval_every} updates, {cfg.eval.eval_games} games '
      f'vs {cfg.eval.eval_opponents}')
if ENABLE_BC:
    print(f'\nBC pretrain: {cfg.imitation.bc_games} demo games '
          f'({cfg.imitation.bc_expert} vs {cfg.imitation.bc_demo_opponent}), '
          f'{cfg.imitation.bc_epochs} epochs')
    print(f'Imitation coef: {cfg.imitation.coef_start} -> 0 '
          f'over {cfg.imitation.coef_decay_updates} PPO updates')
    print(f'Distilled opponent: {cfg.imitation.distilled_opponent}')
else:
    print(f'\nBC pretrain: disabled (pure PPO)')

CHECKPOINT_DIR = f'{cfg.save_dir}/{cfg.run_name}'

In [ ]:
#@title 4. Train

import yaml, sys, os
sys.path.insert(0, '/content/orbit_wars')
os.chdir('/content/orbit_wars')

# ── Resume from checkpoint? ──────────────────────────────────────────────
# Set to a checkpoint path to resume training after a crash/disconnect.
# Leave as None for a fresh start.
RESUME_FROM = None  # e.g. f'{DRIVE_OUTPUT}/checkpoints/{cfg.run_name}/ckpt_last.pt'

# Write full config to temp YAML using the serialization helper
from v2.config import v2_config_to_dict
temp_cfg = '/tmp/v2_train_cfg.yaml'
with open(temp_cfg, 'w') as f:
    yaml.dump(v2_config_to_dict(cfg), f)

resume_flag = f' --resume {RESUME_FROM}' if RESUME_FROM else ''
!python -m v2.train --config {temp_cfg}{resume_flag}

print(f'\nCheckpoints saved to: {CHECKPOINT_DIR}')

## Save Offline Data

Save the demonstration buffer and collect additional apex vs apex games for future offline training.
This data can be reused across sessions without re-collecting.

In [ ]:
#@title 5. Save Offline Data for Future Training

import numpy as np
import pickle
import time
from pathlib import Path

offline_dir = Path(DRIVE_OUTPUT) / 'offline_data'
offline_dir.mkdir(parents=True, exist_ok=True)

# ── 1. Save BC demo buffer (apex vs random) ─────────────────────────────
# This is the same data used for BC pretrain — save it so we don't
# have to re-collect next time.
print('=== Collecting apex vs random demonstrations ===')
from v2.imitation import collect_v2_demonstrations, V2DemonstrationBuffer

t0 = time.time()
demo_buffer_random = collect_v2_demonstrations(
    n_games=200,
    cfg=cfg,
    opponent_name='random',
)
print(f'  Collected in {time.time() - t0:.0f}s')

# Save as numpy arrays
demo_path = offline_dir / 'demo_apex_vs_random.npz'
np.savez_compressed(
    demo_path,
    planet_features=np.array(demo_buffer_random.planet_features),
    global_features=np.array(demo_buffer_random.global_features),
    planet_mask=np.array(demo_buffer_random.planet_mask),
    own_mask=np.array(demo_buffer_random.own_mask),
    reachability_mask=np.array(demo_buffer_random.reachability_mask),
    target_indices=np.array(demo_buffer_random.target_indices),
    ship_fractions=np.array(demo_buffer_random.ship_fractions),
)
print(f'  Saved {len(demo_buffer_random)} samples to {demo_path}')
print(f'  File size: {demo_path.stat().st_size / 1e6:.1f} MB')

# ── 2. Collect apex vs apex demonstrations ───────────────────────────────
# Higher-quality competitive play data.
print('\n=== Collecting apex vs apex demonstrations ===')
t0 = time.time()
demo_buffer_apex = collect_v2_demonstrations(
    n_games=100,
    cfg=cfg,
    opponent_name='apex',
)
print(f'  Collected in {time.time() - t0:.0f}s')

demo_path2 = offline_dir / 'demo_apex_vs_apex.npz'
np.savez_compressed(
    demo_path2,
    planet_features=np.array(demo_buffer_apex.planet_features),
    global_features=np.array(demo_buffer_apex.global_features),
    planet_mask=np.array(demo_buffer_apex.planet_mask),
    own_mask=np.array(demo_buffer_apex.own_mask),
    reachability_mask=np.array(demo_buffer_apex.reachability_mask),
    target_indices=np.array(demo_buffer_apex.target_indices),
    ship_fractions=np.array(demo_buffer_apex.ship_fractions),
)
print(f'  Saved {len(demo_buffer_apex)} samples to {demo_path2}')
print(f'  File size: {demo_path2.stat().st_size / 1e6:.1f} MB')

# ── 3. Summary ───────────────────────────────────────────────────────────
print(f'\n=== Offline Data Summary ===')
for p in sorted(offline_dir.glob('*.npz')):
    data = np.load(p)
    print(f'  {p.name}: {len(data["target_indices"])} samples, '
          f'{p.stat().st_size / 1e6:.1f} MB')
print(f'\nTo load in a future session:')
print(f"  data = np.load('{offline_dir}/demo_apex_vs_random.npz')")
print(f"  # data['planet_features'], data['target_indices'], etc.")

## Submission

Copy the trained V2 checkpoint and the self-contained agent to Drive for Kaggle submission.

In [ ]:
#@title 6. Generate Submission

import shutil
from pathlib import Path

# ── Copy V2 agent + checkpoint to submissions dir ─────────────────────────
sub_dir = Path(DRIVE_OUTPUT) / 'submissions' / 'v2'
sub_dir.mkdir(parents=True, exist_ok=True)

# Copy checkpoint
ckpt_src = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
if ckpt_src.exists():
    shutil.copy2(ckpt_src, sub_dir / 'ckpt_last.pt')
    print(f'Checkpoint copied to: {sub_dir / "ckpt_last.pt"}')
else:
    print(f'No checkpoint found at {ckpt_src}')

# Build self-contained submission file
# Read v2/agent.py and v2/model.py, combine into single file
agent_src = Path('v2/agent.py').read_text()
model_src = Path('v2/model.py').read_text()
policy_block_src = Path('src/policy.py').read_text()

# Extract TransformerBlock class from src/policy.py
import re
tb_match = re.search(
    r'(class TransformerBlock\(nn\.Module\):.*?)(?=\nclass |\Z)',
    policy_block_src, re.DOTALL
)
transformer_block_code = tb_match.group(1) if tb_match else ''

# Extract OrbitNet + OrbitNetOutput from v2/model.py (skip imports)
model_lines = []
skip_imports = True
for line in model_src.split('\n'):
    if skip_imports and (line.startswith('from ') or line.startswith('import ') or
                         line.startswith('"""') or line.strip() == ''):
        continue
    skip_imports = False
    model_lines.append(line)
model_body = '\n'.join(model_lines)

# Build combined submission
submission_content = f'''"""V2 OrbitNet submission for Orbit Wars."""
from __future__ import annotations

import math
import os
from dataclasses import dataclass
from typing import Any

import numpy as np
import torch
import torch.nn as nn

# ── TransformerBlock (from src/policy.py) ──────────────────────────────────
{transformer_block_code}

# ── V2ModelConfig ─────────────────────────────────────────────────────────
@dataclass(slots=True)
class V2ModelConfig:
    embed_dim: int = 128
    n_heads: int = 4
    n_layers: int = 3
    ff_dim: int = 256
    planet_feat_dim: int = 22
    global_feat_dim: int = 8

# ── OrbitNet (from v2/model.py) ───────────────────────────────────────────
{model_body}
'''

# Append the agent code (already self-contained)
# Remove its own imports of v2.model and v2.config since they're inlined above
agent_lines = []
for line in agent_src.split('\n'):
    if 'from v2.model import' in line or 'from v2.config import' in line:
        continue
    agent_lines.append(line)
submission_content += '\n'.join(agent_lines)

submission_path = sub_dir / 'submission.py'
submission_path.write_text(submission_content)
print(f'Submission written to: {submission_path}')
print(f'Submission size: {len(submission_content)} chars')

print(f'\nTo submit to Kaggle:')
print(f'  1. Download {sub_dir} folder')
print(f'  2. Upload submission.py + ckpt_last.pt as a Kaggle dataset')
print(f'  3. Submit submission.py')

## Evaluation

Run the trained V2 OrbitNet head-to-head against apex and random agents.

In [ ]:
#@title 7. Evaluate Trained Model

import torch
from pathlib import Path
from v2.config import load_v2_config
from v2.model import OrbitNet
from v2.train import make_v2_eval_agent
from evaluation.evaluate import run_games, print_results

# Load checkpoint
ckpt_path = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
if not ckpt_path.exists():
    print(f'No checkpoint found at {ckpt_path}')
    print('Available checkpoints:')
    for p in sorted(Path(DRIVE_OUTPUT, 'checkpoints').rglob('ckpt_last.pt')):
        print(f'  {p}')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = OrbitNet(cfg.model).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print(f'Loaded checkpoint: {ckpt_path} (update {ckpt["update"]})')

    eval_agent = make_v2_eval_agent(model, cfg, device)

    N_GAMES = 20

    from agents.apex import agent as apex_agent
    print(f'\nV2 vs Apex ({N_GAMES} games):')
    r = run_games(eval_agent, apex_agent, n_games=N_GAMES, verbose=True)
    print_results('v2_trained', 'apex', r)

    from kaggle_environments.envs.orbit_wars.orbit_wars import random_agent
    print(f'\nV2 vs Random ({N_GAMES} games):')
    r2 = run_games(eval_agent, random_agent, n_games=N_GAMES, verbose=True)
    print_results('v2_trained', 'random', r2)

## Monitoring

TensorBoard shows train/\* and eval/\* metrics.
**Note:** TensorBoard may block execution of cells below it — run this last.

In [ ]:
#@title 8. TensorBoard

%load_ext tensorboard
%tensorboard --logdir {DRIVE_OUTPUT}/logs